# Environment and Reproducibility

## Scientific objective
Record the exact runtime, dependency versions, deterministic seed behavior, and hardware availability.

## Inputs
- `environment.yml`
- `requirements.txt`
- `pyproject.toml`

## Expected outputs
- `reports/environment_snapshot.json`
- `reports/reproducibility_seed_check.json`

## Dependencies
Python standard library, NumPy, PyTorch (optional hardware audit)

## Reproducibility seed
`20260723`. The seed is loaded from `configs/training_config.yaml`; split files and checkpoints are persisted.

## Data and model assumptions
Pinned project files are the declared environment; the current notebook records deviations rather than hiding them.

## Validation checks
The executable cells below fail explicitly on missing/inconsistent required artifacts and save machine-readable status records.

## Interpretation of results
Interpret endpoint-level outputs only after checking prevalence, missingness, split integrity, calibration, uncertainty, and applicability-domain coverage. No notebook result is evidence that experimental toxicity testing can be replaced.

## Saved artifacts
Artifacts listed above are written under `data/`, `models/`, `results/`, `figures/`, `tables/`, or `reports/` and are consumed by later notebooks.

## Limitations
Some GPU kernels can remain nondeterministic despite deterministic settings; such deviations must be disclosed.

## Next notebook
[02_dataset_acquisition.ipynb](./02_dataset_acquisition.ipynb)

In [1]:
from pathlib import Path
import os, json, warnings
import numpy as np
import pandas as pd

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if not (ROOT / "pyproject.toml").exists():
    raise RuntimeError("Run this notebook from the repository root or notebooks directory")
os.chdir(ROOT)

from toxicity_screening.config import load_configs, execution_profile
from toxicity_screening.utils import set_global_seed, require_paths

CONFIGS = load_configs(ROOT)
PROFILE, PROFILE_CONFIG = execution_profile(CONFIGS)
SEED = int(CONFIGS["training_config"]["seed"])
set_global_seed(SEED)
print({"root": str(ROOT), "profile": PROFILE, "seed": SEED})

{'root': 'D:\\Dropbox\\Work\\Learning\\Python\\toxicity_screening_project', 'profile': 'smoke', 'seed': 20260723}


In [2]:
from toxicity_screening.utils import environment_snapshot, atomic_write_json
snapshot = environment_snapshot(ROOT / "reports/environment_snapshot.json")
snapshot

{'created_at': '2026-07-25T21:26:48.639405+00:00',
 'python': '3.11.10 | packaged by conda-forge | (main, Oct 16 2024, 01:17:14) [MSC v.1941 64 bit (AMD64)]',
 'platform': 'Windows-10-10.0.26200-SP0',
 'executable': 'D:\\Users\\anaconda3\\envs\\toxicity-screening\\python.exe',
 'packages': {'numpy': '1.26.4',
  'pandas': '2.2.3',
  'scipy': '1.14.1',
  'rdkit': '2023.9.6',
  'scikit-learn': '1.5.2',
  'xgboost': '2.1.3',
  'torch': '2.5.1',
  'torch-geometric': '2.6.1',
  'optuna': '4.1.0',
  'shap': '0.46.0',
  'captum': '0.7.0',
  'pytdc': '1.1.15',
  'matplotlib': '3.9.2',
  'joblib': '1.4.2',
  'PyYAML': '6.0.2'},
 'environment': {'CONDA_PREFIX': 'D:\\Users\\anaconda3',
  'CUDA_VISIBLE_DEVICES': None,
  'TOX_SCREEN_PROFILE': None}}

In [3]:
# Determinism check for the project seed.
set_global_seed(SEED)
a = np.random.random(8)
set_global_seed(SEED)
b = np.random.random(8)
check = {"seed": SEED, "numpy_reproducible": bool(np.array_equal(a, b))}
try:
    import torch
    check.update({"torch_version": torch.__version__, "cuda_available": torch.cuda.is_available()})
except ImportError:
    check.update({"torch_version": None, "cuda_available": False})
atomic_write_json(check, ROOT / "reports/reproducibility_seed_check.json")
assert check["numpy_reproducible"]
check

{'seed': 20260723,
 'numpy_reproducible': True,
 'torch_version': '2.5.1+cpu',
 'cuda_available': False}

### Completion gate
Confirm that the declared artifacts exist before continuing to `02_dataset_acquisition.ipynb`.